In [2]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,START,END

from pydantic import BaseModel,Field
from typing import Literal
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


In [8]:
import os 
file1="./medical_report.pdf"
loader=PyPDFLoader(file1)
docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
doc1=splitter.split_documents(docs)

key1=os.getenv("Gemini_KEY")

embeddis=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=key1
)
vector=embeddis.embed_query(doc1[0].page_content)



chm=Chroma.from_documents(
   documents=doc1,
   embedding=embeddis,
   persist_directory="./.db_medical1"
)

print("Everthing done")




Everthing done


In [22]:
class RAGS(BaseModel):
    question:str=Field(description="Users Question")
    documents:list=[]
    context:str= Field(description="Context Data for users Questions",default="")
    answer:str=Field(description="Users Question",default="")

key2=os.getenv("GroqKEY")
llm=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=key2)
     

In [23]:
def ragnode(state:RAGS) -> RAGS:
    doc2=chm.similarity_search(query=state.question)
    state.documents=doc2
    return state

def create_context(state:RAGS) -> RAGS:
    context=""
    for doc in state.documents:
        context+=doc.page_content+"\n\n"
    state.context=context
    return state

def gene_node(state:RAGS)->RAGS:
    prompt=f"""
      You are a helpful agent and you will answer the question based on the context provided.
      if you dont find the relevenat info then return as I dont know or i cant get it 
      Context:{state.context}
      Question:{state.question}
    """
    res=llm.invoke(prompt)
    state.answer=res.content
    return state

In [25]:
gph=StateGraph(RAGS)

gph.add_node("ragnode",ragnode)
gph.add_node("create_context",create_context)
gph.add_node("generate_node",gene_node)

gph.add_edge(START,"ragnode")
gph.add_edge("ragnode","create_context")
gph.add_edge("create_context","generate_node")
gph.add_edge("generate_node",END)


graph = gph.compile()

In [28]:
while True:
   query=input("User: ")
   if query.lower() in ["exit","end","bye","quit"]:
     break
   res = graph.invoke({"question":query})
   print(res["answer"])

I don't know.
I don’t know.
The doctor named in the report is **Dr. Nitin Nahar**.
**Summary of the laboratory report**

| Test group | Parameter | Result | Reference range | Comment |
|------------|-----------|--------|-----------------|---------|
| **Liver panel** | AST (SGOT) | 16.5 U/L | < 35 U/L | Normal |
| | ALT (SGPT) | 13.3 U/L | < 35 U/L | Normal |
| | Alkaline phosphatase | 55.9 U/L | 30–120 U/L | Normal |
| | Bilirubin total | 1.06 mg/dL | 0.30–1.20 mg/dL | Normal |
| | Bilirubin direct | 0.18 mg/dL | < 0.20 mg/dL | Normal |
| | Bilirubin indirect | 0.88 mg/dL | < 1.10 mg/dL | Normal |
| | Total protein | 7.49 g/dL | 6.40–8.30 g/dL | Normal |
| | Albumin | 4.60 g/dL | 3.50–5.20 g/dL | Normal |
| | Globulin (calculated) | 2.89 g/dL | 2.0–3.5 g/dL | Normal |
| | A/G ratio | 1.59 | 0.90–2.00 | Normal |
| **Kidney panel** | Creatinine | 0.73 mg/dL | 0.51–0.95 mg/dL | Normal |
| **Hemogram** | Hemoglobin | 13.10 g/dL | 12.0–15.0 g/dL | Normal |
| | PCV (Hematocrit) | 39.5 % | 36